# Pay for Secure Data (x402)

## Overview

**Amazon Bedrock AgentCore payments** enables AI agents to make autonomous
payments for digital services — without ever holding private keys or requiring
human approval for each transaction.

This use case builds a Strands agent that pays for **secure** access to a
metered x402 service. Before the agent pays a registered target x402 service,
it first pays **t54 x402-secure** to risk-score that exact endpoint. Only if
the score clears the configured threshold (and the endpoint is not flagged as a
scam) does the agent go on to pay the target and return the content. The
included target is the **Heurist YahooFinanceAgent** market-data endpoint.

Both the trust check and the target call are paid x402 calls. When either
endpoint returns HTTP 402, the `AgentCorePaymentsPlugin` forwards the payment
requirement to AgentCore payments' `ProcessPayment` operation, receives a
signed proof, and retries transparently. The trust decision is enforced **in
code** by `TrustedX402ServiceGateway`, not by the model's prompt.

### Use Case Details

| Information         | Details                                                               |
|:--------------------|:----------------------------------------------------------------------|
| Use case type       | Trust-gated paid x402 service access with autonomous micropayment     |
| AgentCore components| Amazon Bedrock AgentCore payments, AgentCore Runtime                  |
| Wallet providers    | Coinbase CDP ✅                                                       |
| Payment protocol    | x402 (HTTP 402 Payment Required) on the wire                          |
| Guardrail service   | t54 x402-secure direct API                                            |
| Included target     | Heurist YahooFinanceAgent x402 endpoint                              |
| Agent type          | Single                                                                |
| Agentic Framework   | Strands Agents                                                        |
| LLM model           | Anthropic Claude Sonnet 4.5 (Amazon Bedrock, `us.` inference profile) |
| Example complexity  | Intermediate                                                          |
| SDK used            | boto3                                                                 |

### Architecture

<div style="text-align:left">
    <img src="images/architecture_pay_for_x402_secure_data.png" alt="Pay for Secure Data (x402) architecture — a swimlane diagram across five lanes: Application backend, AgentCore Runtime container, Strands agent plus deterministic trust gate, External paid services, and AgentCore payments. The application backend sends a prompt plus payment context to the runtime's /invocations endpoint; a PaymentContext extractor reads the user id, session, and instrument, and a request-scoped trust-state ContextVar with a TTL is created. The Strands agent calls check_x402_endpoint_trust against the exact target URL; t54 x402-secure (POST /x402/tools/get_overall_score) returns HTTP 402, AgentCorePaymentsPlugin calls ProcessPayment to sign an x402 proof from the session spending limit and retries, and the trust result (overall_score, risk level, scam flag) is stored. TrustedX402ServiceGateway is a fail-closed guardrail with a Trust pass? decision: on fail it returns a blocked result with no target payment; on pass it calls the registered Heurist YahooFinanceAgent x402 service, which may also return HTTP 402 handled by the same plugin, and paid data is returned to the application with the trust result attached." width="85%"/>
</div>

**Numbered flow (matches the diagram)**

1. The **application backend** invokes **AgentCore Runtime** (`POST /invocations`) with a prompt plus `user_id`, `payment_session_id`, and `payment_instrument_id`.
2. The runtime's `PaymentContext` extractor reads that context, opens a **request-scoped trust state** (a `ContextVar` with a TTL), and starts the Strands agent turn.
3. The agent calls `check_x402_endpoint_trust` against the exact registered target endpoint URL.
4. **t54 x402-secure** (`POST /x402/tools/get_overall_score`) returns **HTTP 402** when payment is required; `AgentCorePaymentsPlugin` calls **AgentCore payments** `ProcessPayment`, attaches the signed x402 proof (drawn from the session spending limit), and retries the trust check.
5. The successful trust response (`overall_score`, `risk_level`, `is_scam`) is stored in the request-scoped trust state.
6. `TrustedX402ServiceGateway` (fail-closed) validates the requested `service_id`, `operation`, payload, and cached trust result before any target service payment.
7. If trust is missing, stale, low-score, scam-flagged, or URL-mismatched, the tool returns a **blocked** result and no target payment starts.
8. If trust passes, the gateway calls the registered target x402 service (**Heurist YahooFinanceAgent**).
9. The target service can also return **HTTP 402**; the same payments plugin calls `ProcessPayment`, attaches the signed proof, and retries.
10. Paid data is returned to the **application** with the trust result attached.

The management role creates payment sessions and instruments; the runtime payment role only processes payments. Session spending limits and payment-instrument scope are enforced by AgentCore payments, not by agent prompt instructions. t54 x402-secure supplies the pre-payment trust signal, and `call_trusted_x402_service` enforces the trust decision in code before a target x402 payment can start — the prompt instructs the order, but the gateway blocks missing, expired, low-score, scam, or URL-mismatched trust state deterministically.

### Use Case Key Features

* Agent is designed not to hold private keys — AgentCore payments signs every charge
* Pre-payment trust guardrail — t54 x402-secure scores the target before the agent pays it
* Deterministic, code-enforced gate — blocks missing/expired/low-score/scam/URL-mismatched trust before any target payment
* Human-controlled spending limit via `maxSpendAmount` — one session spending limit covers both paid calls
* IAM role separation: `ManagementRole` creates sessions, `ProcessPaymentRole` signs payments (explicit `Deny` both ways)
* Per-invocation payment context — one runtime deployment serves many users and sessions


## Prerequisites

To run this notebook you will need:

* **AWS account** with Amazon Bedrock AgentCore payments available in your chosen region
* **Amazon Bedrock** access enabled for **Anthropic Claude Sonnet 4.5** in your chosen region
* **Python 3.10+** and a Jupyter kernel (JupyterLab, classic Notebook, VS Code, or Kiro)
* **AWS CLI v2** configured with credentials (`aws configure`) and **`jq`**
* **AWS CDK v2** (`npm install -g aws-cdk`) and **Node.js 18+** — used by §8 to deploy the agent runtime
* **AgentCore payments botocore service definitions** available to your boto3 install. If your account is in the preview, install the current control- and data-plane service models into `~/.aws/models`; an outdated model fails on the `EMBEDDED_CRYPTO_WALLET` instrument type. Verify with `aws bedrock-agentcore-control help`.
* **A Coinbase Developer Platform (CDP) account** — API Key ID, API Key Secret, Wallet Secret
* **A real, deliverable email** for `INSTRUMENT_EMAIL` — the per-wallet delegated-signing grant sends a one-time code there
* **USDC on Base** to fund the embedded wallet §5 creates. This use case calls **live external x402 services** (t54 x402-secure and Heurist), which settle real payments — keep the session spending limit small.

> ⚠️ **Real-money notice.** Unlike samples that deploy their own testnet seller,
> §7 and §8 call live external x402 services that settle **real USDC twice** per
> approved run. §3 lets you exercise the guardrail with **mocks and no money**
> first. The live run in §7 is gated behind an explicit `RUN_LIVE` flag.


## 1. Install dependencies

Run the cell below to install every Python dependency the notebook and the
agent need.


In [ ]:
# Use %pip (not !pip) so packages install into THIS kernel's Python
# environment, not whatever pip happens to be on the shell PATH.
%pip install -r requirements.txt --quiet

## 2. Configure environment

The notebook reads everything from a `.env` file in this folder. Run the cell
below to:

1. Create the four IAM roles the notebook assumes into (idempotent — safe to
   re-run). `setup-roles.sh` writes the role ARNs directly into `.env`.
2. Open `.env` in an editor tab and list which values you still need to paste
   in. Save the file, then **re-run the Environment check cell** to continue.

You need values for:

- `AWS_REGION` — an AgentCore payments preview region (seeded as `us-west-2`).
- `CONFIRM_AWS_ACCOUNT_ID` — your 12-digit account ID. `setup-roles.sh` refuses
  to create roles unless this matches the current caller.
- `COINBASE_API_KEY_ID`, `COINBASE_API_KEY_SECRET`, `COINBASE_WALLET_SECRET` —
  from coinbase.com/developer-platform → Project → API Keys + Wallet. Enable
  *Delegated signing* under Project → Wallet → Embedded Wallets → Policies.
- `INSTRUMENT_EMAIL` — a **real** inbox you control. The Coinbase Wallet Hub
  sends a one-time code there when you grant the per-wallet delegation in §5.

`USER_ID` is auto-generated for you the first time you run this cell.


In [ ]:
# Step 1: create the IAM roles. setup-roles.sh verifies CONFIRM_AWS_ACCOUNT_ID
# against the caller and writes the four role ARNs into .env (idempotent).
import shutil
import subprocess
from pathlib import Path

USE_CASE = Path(".").resolve()
ENV_FILE = USE_CASE / ".env"
TEMPLATE = USE_CASE / "env-sample.txt"

if not ENV_FILE.exists():
    shutil.copy2(TEMPLATE, ENV_FILE)
    print(f"✅ Seeded {ENV_FILE.name} from {TEMPLATE.name}")

# Seed USER_ID before role creation so the .env is complete.
subprocess.run(["bash", "test/integration/setup-env.sh"], check=False)

# Remind users which keys they must fill in before roles can be created.
REQUIRED_MANUAL = [
    ("CONFIRM_AWS_ACCOUNT_ID", "Your 12-digit AWS account ID (setup-roles.sh checks it against the caller)"),
    ("COINBASE_API_KEY_ID", "Coinbase CDP API key ID (coinbase.com/developer-platform → Project → API Keys)"),
    ("COINBASE_API_KEY_SECRET", "Coinbase CDP API key secret"),
    ("COINBASE_WALLET_SECRET", "Coinbase CDP wallet secret (Project → Wallet; enable Delegated signing first)"),
    ("INSTRUMENT_EMAIL", "Real inbox you control — Coinbase Wallet Hub sends a one-time code here"),
]

env_values = {}
for line in ENV_FILE.read_text().splitlines():
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1)
        env_values[k.strip()] = v.strip()

missing = [(k, h) for k, h in REQUIRED_MANUAL if not env_values.get(k) or env_values[k].startswith("<")]

if missing:
    print(f"\n⏳ Fill in these {len(missing)} values in {ENV_FILE.name}, then run the roles step again:\n")
    for key, hint in missing:
        print(f"   • {key}")
        print(f"       {hint}")
    try:
        subprocess.run(["code", str(ENV_FILE)], check=False)
    except FileNotFoundError:
        print(f"\n   Open manually: {ENV_FILE}")
else:
    print("\n✅ Required manual values are set. Creating IAM roles...\n")
    roles_proc = subprocess.run(["bash", "test/integration/setup-roles.sh"], check=False)
    if roles_proc.returncode != 0:
        raise RuntimeError(
            f"setup-roles.sh exited {roles_proc.returncode} — fix the error above and re-run this cell."
        )

In [ ]:
# Environment check — run after .env is fully filled in.
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")

# IAM role ARNs (from setup-roles.sh)
CONTROL_PLANE_ROLE_ARN = os.environ.get("CONTROL_PLANE_ROLE_ARN", "")
MANAGEMENT_ROLE_ARN = os.environ.get("MANAGEMENT_ROLE_ARN", "")
PROCESS_PAYMENT_ROLE_ARN = os.environ.get("PROCESS_PAYMENT_ROLE_ARN", "")
RESOURCE_RETRIEVAL_ROLE_ARN = os.environ.get("RESOURCE_RETRIEVAL_ROLE_ARN", "")

# Coinbase CDP secrets
COINBASE_API_KEY_ID = os.environ.get("COINBASE_API_KEY_ID", "")
COINBASE_API_KEY_SECRET = os.environ.get("COINBASE_API_KEY_SECRET", "")
COINBASE_WALLET_SECRET = os.environ.get("COINBASE_WALLET_SECRET", "")

# Instrument + session config
import uuid as _uuid

USER_ID = os.environ.get("USER_ID") or f"x402-secure-data-{_uuid.uuid4()}"
INSTRUMENT_EMAIL = os.environ.get("INSTRUMENT_EMAIL", "")
PAYMENT_INSTRUMENT_NETWORK = os.environ.get("PAYMENT_INSTRUMENT_NETWORK", "ETHEREUM")
PAYMENT_SESSION_MAX_SPEND_USD = os.environ.get("PAYMENT_SESSION_MAX_SPEND_USD", "1.0")
PAYMENT_SESSION_EXPIRY_MINUTES = int(os.environ.get("PAYMENT_SESSION_EXPIRY_MINUTES", "60"))

# State populated by §4 / §5
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
PAYMENT_CONNECTOR_ID = os.environ.get("PAYMENT_CONNECTOR_ID", "")
CREDENTIAL_PROVIDER_ARN = os.environ.get("CREDENTIAL_PROVIDER_ARN", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")
PAYMENT_SESSION_ID = os.environ.get("PAYMENT_SESSION_ID", "")
WALLET_ADDRESS = os.environ.get("WALLET_ADDRESS", "")


def _check(label, value, redact=False, optional=False):
    ok = bool(value) and not value.startswith("<")
    display = "[redacted]" if redact and value else value
    if optional and not ok:
        print(f"  ⏳  {label}: {display or '(will be set later)'}")
        return True
    print(f"  {'✅' if ok else '❌ MISSING'}  {label}: {display}")
    return ok


_results = []


def _req(label, value, redact=False):
    _results.append(_check(label, value, redact=redact))


print("=== Environment check ===")
print("\n  AWS:")
_req("AWS_REGION", AWS_REGION)
print("\n  IAM roles (from setup-roles.sh):")
_req("CONTROL_PLANE_ROLE_ARN", CONTROL_PLANE_ROLE_ARN)
_req("MANAGEMENT_ROLE_ARN", MANAGEMENT_ROLE_ARN)
_req("PROCESS_PAYMENT_ROLE_ARN", PROCESS_PAYMENT_ROLE_ARN)
_req("RESOURCE_RETRIEVAL_ROLE_ARN", RESOURCE_RETRIEVAL_ROLE_ARN)
print("\n  Coinbase CDP secrets:")
_req("COINBASE_API_KEY_ID", COINBASE_API_KEY_ID)
_req("COINBASE_API_KEY_SECRET", COINBASE_API_KEY_SECRET, redact=True)
_req("COINBASE_WALLET_SECRET", COINBASE_WALLET_SECRET, redact=True)
print("\n  Wallet linked email (used by §5 CreatePaymentInstrument):")
_req("INSTRUMENT_EMAIL", INSTRUMENT_EMAIL)
print("\n  Populated later — ⏳ is expected on first run:")
_check("MANAGER_ARN", MANAGER_ARN, optional=True)
_check("PAYMENT_INSTRUMENT_ID", PAYMENT_INSTRUMENT_ID, optional=True)
_check("PAYMENT_SESSION_ID", PAYMENT_SESSION_ID, optional=True)

if INSTRUMENT_EMAIL.lower().endswith("@example.com"):
    raise SystemExit("INSTRUMENT_EMAIL must be a REAL, deliverable address — not an @example.com placeholder.")
if not all(_results):
    raise SystemExit(
        f"{_results.count(False)} required value(s) missing. Fill them in .env, save, then re-run this cell."
    )
print("\n✅ Environment looks good.")

## 3. See the trust guardrail in action (no AWS, no money)

Before spending anything, exercise the deterministic guardrail with **mocked**
trust and service clients. This makes no AWS calls and settles no payments — it
just shows how `TrustedX402ServiceGateway` decides to approve or block a target
payment based on the trust result.

The gateway is what makes the guardrail non-bypassable: the agent's prompt
orders the two tools, but the gateway enforces the decision in code. Trust
results are **request-scoped** — a score obtained in one request never
authorizes a later one.


In [ ]:
import sys

sys.path.insert(0, str((USE_CASE / "agent" / "container").resolve()))

from x402_services import (
    DEFAULT_SERVICE_ID,
    HEURIST_YAHOO_FINANCE_OPERATIONS,
    TrustedX402ServiceGateway,
    resolve_heurist_base_url,
    use_request_trust_state,
)

TARGET_URL = resolve_heurist_base_url()


class DemoTrustClient:
    """Stand-in for t54 x402-secure — returns a fixed score, no HTTP, no payment."""

    def __init__(self, result):
        self.result = result

    def score_endpoint(self, url, headers=None):
        return self.result


class DemoServiceClient:
    """Stand-in for the target x402 service — returns fake data, no HTTP, no payment."""

    def call_operation(self, operation, payload, headers=None):
        return {"operation": operation, "payload": payload, "data": "<mocked market data>"}


def make_gateway(trust_result):
    services = {
        DEFAULT_SERVICE_ID: {
            "base_url": TARGET_URL,
            "operations": HEURIST_YAHOO_FINANCE_OPERATIONS,
            "client": DemoServiceClient(),
        }
    }
    return TrustedX402ServiceGateway(
        services=services,
        trust_client=DemoTrustClient(trust_result),
        trust_threshold=50,
        fail_closed=True,
    )


def demo(label, trust_result):
    print(f"\n── {label} ──")
    with use_request_trust_state():
        gw = make_gateway(trust_result)
        trust = gw.check_x402_endpoint_trust(DEFAULT_SERVICE_ID)
        print(f"  trust score: {trust.get('overall_score')} | risk: {trust.get('risk_level')} | scam: {trust.get('is_scam')}")
        result = gw.call_trusted_x402_service(DEFAULT_SERVICE_ID, "quote_snapshot", {"symbols": "AAPL"})
        if result.get("status") == "blocked":
            print(f"  🚫 BLOCKED — {result['reason']}")
        else:
            print(f"  ✅ target called — {result.get('data')}")


# Approve: high score, not a scam → target service is called.
demo("Approved (score 91)", {"overall_score": 91, "risk_level": "low", "is_scam": False})
# Low score: below threshold 50 → blocked, target never called.
demo("Blocked — low score (score 23)", {"overall_score": 23, "risk_level": "high", "is_scam": False})
# Scam flag: blocked regardless of score.
demo("Blocked — scam flagged", {"overall_score": 88, "risk_level": "critical", "is_scam": True})

print("\nThe target service is only ever reached on the approved path. This same")
print("gateway runs unchanged against the live t54 x402-secure API in §7.")

## 4. Set up AgentCore payments

This section provisions everything AgentCore payments needs, using the Coinbase
CDP provider:

1. **One Credential Provider** — stores the Coinbase CDP secrets in AgentCore Identity.
2. **One Payment Manager** — the top-level payment resource.
3. **One Payment Connector** — binds the Credential Provider to the Manager.

Each step persists its output to `.env` so a kernel restart picks up where you
left off. Re-running is safe — a set ID is detected and the create is skipped.

### 4.1 Assume roles and build clients


In [ ]:
import boto3
from boto3.session import Session

from utils import assume_role, idempotent_create, pp, wait_for_status, write_env_updates

import re
import uuid


def _client_token() -> str:
    # AgentCore payments requires clientToken >= 33 chars (idempotency token).
    return f"{uuid.uuid4()}-{uuid.uuid4().hex[:8]}"


def _safe_name(prefix: str) -> str:
    # Manager/Connector names: start with a letter, letters+digits only, <= 48 chars.
    name = f"{prefix}{uuid.uuid4().hex[:8]}"
    assert re.match(r"^[a-zA-Z][a-zA-Z0-9]{0,47}$", name), name
    return name


for _name, _val in (
    ("CONTROL_PLANE_ROLE_ARN", CONTROL_PLANE_ROLE_ARN),
    ("RESOURCE_RETRIEVAL_ROLE_ARN", RESOURCE_RETRIEVAL_ROLE_ARN),
):
    if not _val:
        raise RuntimeError(f"Missing {_name}. Run `bash test/integration/setup-roles.sh` (or §2).")

boto_session = Session(region_name=AWS_REGION)

print("Assuming ControlPlaneRole...")
cp_session = assume_role(boto_session, CONTROL_PLANE_ROLE_ARN, session_name="x402-secure-cp")
cp_client = cp_session.client("bedrock-agentcore-control")

print("\n✅ Control-plane client ready")

### 4.2 Create the Credential Provider

`PaymentCredentialProvider` is the security handoff. The notebook reads the
Coinbase CDP secrets from `.env` once and calls
`CreatePaymentCredentialProvider` against AgentCore Identity, which stores them
in **AWS Secrets Manager** under **AWS KMS** encryption and surfaces only the
secret ARN to the agent. The agent runtime obtains a short-lived vendor token
through `GetResourcePaymentToken` at signing time and never receives the raw
secret.


In [ ]:
if CREDENTIAL_PROVIDER_ARN:
    print(f"↷ CREDENTIAL_PROVIDER_ARN already set in .env, skipping create")
else:
    missing = [n for n, v in (
        ("COINBASE_API_KEY_ID", COINBASE_API_KEY_ID),
        ("COINBASE_API_KEY_SECRET", COINBASE_API_KEY_SECRET),
        ("COINBASE_WALLET_SECRET", COINBASE_WALLET_SECRET),
    ) if not v]
    if missing:
        raise RuntimeError(f"Missing Coinbase CDP secrets in .env: {missing}")

    CRED_PROVIDER_NAME = _safe_name("CoinbaseCdp")
    resp = idempotent_create(
        cp_client.create_payment_credential_provider,
        conflict_msg=f"Credential provider {CRED_PROVIDER_NAME} already exists",
        name=CRED_PROVIDER_NAME,
        credentialProviderVendor="CoinbaseCDP",
        providerConfigurationInput={
            "coinbaseCdpConfiguration": {
                "apiKeyId": COINBASE_API_KEY_ID,
                "apiKeySecret": COINBASE_API_KEY_SECRET,
                "walletSecret": COINBASE_WALLET_SECRET,
            }
        },
    )
    if resp is None:
        raise RuntimeError("Credential provider already exists — rename or delete it.")
    CREDENTIAL_PROVIDER_ARN = resp["credentialProviderArn"]
    print(f"✅ Coinbase CDP credential provider created")
    write_env_updates({"CREDENTIAL_PROVIDER_ARN": CREDENTIAL_PROVIDER_ARN})
    print("💾 .env updated: CREDENTIAL_PROVIDER_ARN")

### 4.3 Create the Payment Manager

`PaymentManager` is the top-level payment resource. It takes the
`RESOURCE_RETRIEVAL_ROLE_ARN` — the role AgentCore payments assumes at runtime
to retrieve the credentials you stored above.


In [ ]:
if MANAGER_ARN:
    print(f"↷ MANAGER_ARN already set in .env, skipping create:\n   {MANAGER_ARN}")
else:
    MANAGER_NAME = _safe_name("X402SecureData")
    resp = cp_client.create_payment_manager(
        name=MANAGER_NAME,
        description=f"AgentCore payments Pay for Secure Data (x402) {MANAGER_NAME}",
        authorizerType="AWS_IAM",
        roleArn=RESOURCE_RETRIEVAL_ROLE_ARN,
        clientToken=_client_token(),
    )
    MANAGER_ID = resp["paymentManagerId"]
    MANAGER_ARN = resp["paymentManagerArn"]
    print("✅ Payment Manager created")
    print(f"   Manager ARN: {MANAGER_ARN}")

    print("\nWaiting for PaymentManager to reach READY...")
    wait_for_status(
        cp_client.get_payment_manager,
        expected_status="READY",
        paymentManagerId=MANAGER_ID,
    )
    print("✅ PaymentManager is READY")
    write_env_updates({"MANAGER_ARN": MANAGER_ARN})
    print("💾 .env updated: MANAGER_ARN")
MANAGER_ID = MANAGER_ARN.rsplit("/", 1)[-1]

### 4.4 Create the Payment Connector

`PaymentConnector` binds the Credential Provider to the Manager. The
connector's `type` tells AgentCore payments which signer to use.


In [ ]:
if PAYMENT_CONNECTOR_ID:
    print(f"↷ PAYMENT_CONNECTOR_ID already set in .env, skipping create")
else:
    CONNECTOR_NAME = _safe_name("CoinbaseConnector")
    resp = cp_client.create_payment_connector(
        paymentManagerId=MANAGER_ID,
        name=CONNECTOR_NAME,
        description=f"AgentCore payments CoinbaseCDP connector {CONNECTOR_NAME}",
        type="CoinbaseCDP",
        credentialProviderConfigurations=[{"coinbaseCDP": {"credentialProviderArn": CREDENTIAL_PROVIDER_ARN}}],
        clientToken=_client_token(),
    )
    PAYMENT_CONNECTOR_ID = resp["paymentConnectorId"]
    print(f"✅ Coinbase CDP connector: {PAYMENT_CONNECTOR_ID}")

    print("\nWaiting for connector to reach READY...")
    wait_for_status(
        cp_client.get_payment_connector,
        expected_status="READY",
        paymentManagerId=MANAGER_ID,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
    )
    print("✅ Connector is READY")
    write_env_updates({"PAYMENT_CONNECTOR_ID": PAYMENT_CONNECTOR_ID})
    print("💾 .env updated: PAYMENT_CONNECTOR_ID")

## 5. Create the payment instrument and session

`EMBEDDED_CRYPTO_WALLET` is the instrument type. `linkedAccounts` is required —
AgentCore payments uses the email to resolve or create the vendor-side end-user
that owns the wallet. The **management** client (`ManagementRole`) creates the
instrument and the session; it has an explicit `Deny` on `ProcessPayment`.

After the instrument is created, complete the **two-layer delegated signing**
and fund the wallet before running §7 (see the printed steps).

### 5.1 Create the instrument


In [ ]:
from botocore.exceptions import ClientError

mgmt_session = assume_role(boto_session, MANAGEMENT_ROLE_ARN, session_name="x402-secure-mgmt")
dp_client_mgmt = mgmt_session.client("bedrock-agentcore")

if PAYMENT_INSTRUMENT_ID:
    print(f"↷ PAYMENT_INSTRUMENT_ID already set in .env, skipping create")
    REDIRECT_URL = None
else:
    resp = dp_client_mgmt.create_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        userId=USER_ID,
        paymentInstrumentType="EMBEDDED_CRYPTO_WALLET",
        paymentInstrumentDetails={
            "embeddedCryptoWallet": {
                "network": PAYMENT_INSTRUMENT_NETWORK,
                "linkedAccounts": [{"email": {"emailAddress": INSTRUMENT_EMAIL}}],
            }
        },
        clientToken=_client_token(),
    )
    instrument = resp["paymentInstrument"]
    PAYMENT_INSTRUMENT_ID = instrument["paymentInstrumentId"]
    print(f"✅ Payment instrument created: {PAYMENT_INSTRUMENT_ID}")

    print("\nWaiting for instrument to become ACTIVE...")
    wait_for_status(
        dp_client_mgmt.get_payment_instrument,
        expected_status="ACTIVE",
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=USER_ID,
    )
    refreshed = dp_client_mgmt.get_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=USER_ID,
    )["paymentInstrument"]
    crypto = refreshed["paymentInstrumentDetails"]["embeddedCryptoWallet"]
    WALLET_ADDRESS = crypto.get("walletAddress", "")
    REDIRECT_URL = crypto.get("redirectUrl")
    print("✅ Instrument ACTIVE")
    write_env_updates({"PAYMENT_INSTRUMENT_ID": PAYMENT_INSTRUMENT_ID, "WALLET_ADDRESS": WALLET_ADDRESS})
    print("💾 .env updated: PAYMENT_INSTRUMENT_ID, WALLET_ADDRESS")

print("\n" + "=" * 64)
print("  ONE-TIME ONBOARDING — required before §7 can pay")
print("=" * 64)
print("  1. Project delegated signing (once per Coinbase project):")
print("       CDP Portal → Wallets → Non-custodial Wallet → Security →")
print("       enable 'Delegated signing' (requires your account 2FA).")
if REDIRECT_URL:
    print(f"  2. Per-wallet grant: open the Wallet Hub, sign in with INSTRUMENT_EMAIL,")
    print(f"       and Grant signing delegation:\n       {REDIRECT_URL}")
else:
    print("  2. Per-wallet grant: re-fetch the Wallet Hub redirectUrl if not shown,")
    print("       sign in with INSTRUMENT_EMAIL, and Grant signing delegation.")
print(f"  3. Fund the wallet with USDC on {PAYMENT_INSTRUMENT_NETWORK} (Base):")
print(f"       {WALLET_ADDRESS or '(address pending)'}")
print("=" * 64)

### Grant signing delegation in the Coinbase Wallet Hub

The instrument step above prints a `redirectUrl` to the **Coinbase Wallet Hub**.
Delegated signing has **two one-time layers, both required** before
`ProcessPayment` will succeed — the project policy (CDP Portal → Wallets →
Non-custodial Wallet → Security, needs your account 2FA) and this per-wallet
grant. Open the hub and:

1. **Sign in** with the email you set as `INSTRUMENT_EMAIL`. The hub sends a
   one-time passcode (OTP) to that address.

<div style="text-align:left">
    <img src="images/cdp_hub_signin.png" alt="Coinbase Wallet Hub sign-in screen" width="60%"/>
</div>

2. **Enter the OTP** in the hub.

<div style="text-align:left">
    <img src="images/cdp_hub_otp.png" alt="Coinbase Wallet Hub OTP entry" width="60%"/>
</div>

3. **Grant signing delegation** to the agent and set the delegation duration.
   Without this, `ProcessPayment` returns *Delegated signing grant is not
   active*.

<div style="text-align:left">
    <img src="images/cdp_hub_delegation.png" alt="Grant delegation with duration" width="60%"/>
</div>

4. **Copy the wallet address** printed above and fund it with **USDC on Base**.
   Both the t54 x402-secure trust check and the target call settle real USDC, so
   the wallet must be funded before §7.


### 5.2 Create the payment session

In [ ]:
if PAYMENT_SESSION_ID:
    print(f"↷ PAYMENT_SESSION_ID already set in .env, skipping create")
else:
    resp = dp_client_mgmt.create_payment_session(
        paymentManagerArn=MANAGER_ARN,
        userId=USER_ID,
        expiryTimeInMinutes=PAYMENT_SESSION_EXPIRY_MINUTES,
        limits={"maxSpendAmount": {"value": str(PAYMENT_SESSION_MAX_SPEND_USD), "currency": "USD"}},
        clientToken=_client_token(),
    )
    PAYMENT_SESSION_ID = resp["paymentSession"]["paymentSessionId"]
    print("✅ Payment session created")
    print(f"   Session ID: {PAYMENT_SESSION_ID}")
    print(f"   Budget:     ${PAYMENT_SESSION_MAX_SPEND_USD} USD (covers the trust check + target call)")
    print(f"   Expires in: {PAYMENT_SESSION_EXPIRY_MINUTES} minutes")
    write_env_updates({"PAYMENT_SESSION_ID": PAYMENT_SESSION_ID})
    print("💾 .env updated: PAYMENT_SESSION_ID")

## 6. Build the trust-gated agent (local)

The agent has exactly two tools — `check_x402_endpoint_trust` and
`call_trusted_x402_service` — plus the `AgentCorePaymentsPlugin` for automatic
x402 handling. The plugin config is resolved from the environment
(`MANAGER_ARN`, `PAYMENT_SESSION_ID`, `PAYMENT_INSTRUMENT_ID`, `USER_ID`), which
`.env` already holds after §4–§5.

> The local run signs payments with **your notebook's own AWS credentials**, so
> the caller needs `bedrock-agentcore:ProcessPayment`. In §8 the deployed
> runtime uses the dedicated `ProcessPaymentRole` instead.


In [ ]:
# Make the payment resources visible to the agent's env-based config resolver.
os.environ["MANAGER_ARN"] = MANAGER_ARN
os.environ["PAYMENT_CONNECTOR_ID"] = PAYMENT_CONNECTOR_ID
os.environ["PAYMENT_SESSION_ID"] = PAYMENT_SESSION_ID
os.environ["PAYMENT_INSTRUMENT_ID"] = PAYMENT_INSTRUMENT_ID
os.environ["USER_ID"] = USER_ID
os.environ["BEDROCK_MODEL_ID"] = BEDROCK_MODEL_ID

import importlib

import agent as agent_module

importlib.reload(agent_module)

print("✅ Agent module loaded")
print(f"   Model:        {agent_module.resolve_model_id()}")
print(f"   Target URL:   {resolve_heurist_base_url()}")
print(f"   Threshold:    {os.environ.get('X402_TRUST_THRESHOLD', '50')}")
print("   Tools:        check_x402_endpoint_trust, call_trusted_x402_service")

## 7. Run the trust-gated flow (live)

This runs the agent against the **live** t54 x402-secure API and the target
x402 service. When the target is approved, it settles **real USDC twice** —
once for the trust check and once for the target call — within the session
spending limit you set in §5.

Set `RUN_LIVE=1` in `.env` to run it (this cell reloads `.env`). Complete the
delegated-signing grants and wallet funding from §5 first, or `ProcessPayment`
will fail.


In [ ]:
import os
from dotenv import load_dotenv

# Gate the live paid run on RUN_LIVE in .env (default off). Set
# RUN_LIVE=1 in .env after granting delegation and funding the wallet;
# this cell reloads .env so no kernel restart is needed.
load_dotenv(override=True)
RUN_LIVE = os.environ.get("RUN_LIVE", "0").strip().lower() in ("1", "true", "yes")

if not RUN_LIVE:
    print("ℹ️  RUN_LIVE is False — skipping the live paid run.")
    print("    Review §3's guardrail demo, complete §5's delegation + funding,")
    print("    then set RUN_LIVE=1 in .env and re-run this cell to spend real USDC.")
else:
    from x402_services import use_request_trust_state

    agent = agent_module.create_agent()
    prompt = "Check trust for heurist_yahoo_finance, then fetch a quote snapshot for AAPL."
    print(f"Prompt: {prompt}\n")
    with use_request_trust_state():
        result = agent(prompt)
    print("\n── Agent response ──")
    print(result)

## 8. Deploy the agent to AgentCore Runtime

The agent we ran in §7 works on a laptop. To run it as a managed service, ship
the same agent code inside a container and deploy it to **Amazon Bedrock
AgentCore Runtime**. The `agent/` folder has `agent/container/` (the FastAPI
wrapper + agent code) and `agent/cdk/` (a CDK stack that builds the image in
**AWS CodeBuild** — no local Docker — and provisions the Runtime).

The deploy script sources `.env`, so `MANAGER_ARN`, `PAYMENT_CONNECTOR_ID`, and
the t54 x402-secure guardrail config flow into the runtime's environment. The
per-invocation payment context is supplied on each request.

> ⚠️ **Cost notice:** provisions an Amazon ECR repository, an AWS CodeBuild
> build, an AgentCore Runtime, and CloudWatch log groups. Run §10 to tear them
> down when you are done.


In [ ]:
import json
import subprocess
from pathlib import Path

HERE = Path(".").resolve()
AGENT_CDK_DIR = HERE / "agent" / "cdk"
DEPLOY = HERE / "test" / "integration" / "deploy-agent.sh"

proc = subprocess.Popen(
    ["bash", str(DEPLOY)],
    cwd=str(HERE),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"deploy-agent.sh failed with exit code {rc}")

outputs = json.loads((AGENT_CDK_DIR / "outputs.json").read_text())["AgentCorePaymentsX402SecureDataAgentStack"]
AGENT_RUNTIME_ARN = outputs["AgentRuntimeArn"]
AGENT_RUNTIME_ID = outputs["AgentRuntimeId"]
print("\n✅ Runtime deployed")
print(f"   Runtime ARN: {AGENT_RUNTIME_ARN}")

### Enable Transaction Search (one-time, console step)

Before invoking the runtime, enable Transaction Search so the **Observability**
tab on the runtime page shows traces and span details. This is a one-time setup
per runtime.

1. Open the **Amazon Bedrock AgentCore** console → **Runtime**.
2. Choose **`pay_for_x402_secure_data_runtime`** in the list.

<div style="text-align:left">
    <img src="images/agentcore_runtime_selected.png" alt="Selecting the agent runtime in the list" width="75%"/>
</div>

3. Open the **Log deliveries and tracing** tab, enable **Transaction Search**, and choose **Save**.

<div style="text-align:left">
    <img src="images/agentcore_tracing_enable_section.png" alt="Enable Transaction Search and tracing" width="75%"/>
</div>

After saving, the panel confirms Transaction Search is enabled:

<div style="text-align:left">
    <img src="images/agentcore_transaction_search_enabled.png" alt="Transaction Search enabled confirmation" width="75%"/>
</div>

> Transaction Search takes a few minutes to start indexing after you save. If
> the Observability tab shows no data immediately, wait a minute and refresh.


### Invoke the deployed runtime

Invoke with a prompt plus the per-invocation payment context. The container
constructs the agent + plugin per invocation from the payload, so one
deployment serves any user/session/instrument.

> Live invoke — set `RUN_LIVE_RUNTIME = True` to settle real USDC through the
> deployed runtime.


In [ ]:
import json as _json
import os

from dotenv import load_dotenv

# Gate the live runtime invoke on RUN_LIVE_RUNTIME in .env (default off).
load_dotenv(override=True)
RUN_LIVE_RUNTIME = os.environ.get("RUN_LIVE_RUNTIME", "0").strip().lower() in ("1", "true", "yes")

if not RUN_LIVE_RUNTIME:
    print("ℹ️  RUN_LIVE_RUNTIME is False — skipping the live runtime invoke.")
else:
    runtime_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
    payload = {
        "input": {
            "prompt": "Check the registered x402 market-data service, then fetch a quote snapshot for AAPL.",
            "payment_context": {
                "user_id": USER_ID,
                "payment_session_id": PAYMENT_SESSION_ID,
                "payment_instrument_id": PAYMENT_INSTRUMENT_ID,
            },
        }
    }
    resp = runtime_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        qualifier="DEFAULT",
        payload=_json.dumps(payload).encode(),
    )
    body = resp["response"]
    data = b"".join(body.iter_chunks()) if hasattr(body, "iter_chunks") else body.read()
    print(_json.loads(data))

### Inspect the run in the console

After the runtime returns, open the AgentCore console to see the underlying
spans and logs.

1. From the runtime detail page, choose **View dashboard** in the
   **Observability** section.

<div style="text-align:left">
    <img src="images/agentcore_runtime_observability_section.png" alt="Observability section with View dashboard button" width="75%"/>
</div>

2. The CloudWatch GenAI Observability dashboard opens. Open the **Sessions** tab
   and choose the most recent **Session ID**.

<div style="text-align:left">
    <img src="images/agentcore_cloudwatch_genai_observability_dashboard.png" alt="CloudWatch GenAI Observability dashboard, Sessions tab" width="75%"/>
</div>

3. Choose the most recent **Trace ID** to explore the `POST /invocations`
   events: the two tool calls, the payment requirement (`402`), the
   `ProcessPayment` span, and the retry that returns `200 OK` — one pair for the
   t54 x402-secure trust check and one for the target service call.

<div style="text-align:left">
    <img src="images/agentcore_observability_results.png" alt="Trace exploration showing POST /invocations events" width="75%"/>
</div>


## 9. Inspect the data plane

Walk the read-only data-plane APIs to see what the service recorded. These run
against the **management** client — the agent's `ProcessPaymentRole` has an
explicit `Deny` on session/instrument management, which is how the audit
boundary is enforced.


In [ ]:
# §9 is self-contained: it re-reads .env and rebuilds the management client if
# the kernel does not already have it (e.g. you restarted or jumped here after
# the deploy). Runs against the ManagementRole — the agent's ProcessPaymentRole
# has an explicit Deny on these, which is how the audit boundary is enforced.
import os

import boto3
from boto3.session import Session
from botocore.exceptions import ClientError
from dotenv import load_dotenv

from utils import assume_role

load_dotenv(override=True)
AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")
MANAGEMENT_ROLE_ARN = os.environ.get("MANAGEMENT_ROLE_ARN", "")
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
PAYMENT_CONNECTOR_ID = os.environ.get("PAYMENT_CONNECTOR_ID", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")
PAYMENT_SESSION_ID = os.environ.get("PAYMENT_SESSION_ID", "")
USER_ID = os.environ.get("USER_ID", "")
WALLET_ADDRESS = os.environ.get("WALLET_ADDRESS", "")
PAYMENT_INSTRUMENT_NETWORK = os.environ.get("PAYMENT_INSTRUMENT_NETWORK", "ETHEREUM")

# Reuse the management client from §5.1 if present; otherwise assume the role.
try:
    dp_client_mgmt
except NameError:
    if not MANAGEMENT_ROLE_ARN:
        raise SystemExit("MANAGEMENT_ROLE_ARN missing in .env — run §2 (setup-roles.sh) first.")
    dp_client_mgmt = assume_role(
        Session(region_name=AWS_REGION), MANAGEMENT_ROLE_ARN, "x402-secure-mgmt-inspect"
    ).client("bedrock-agentcore")


def resolve_payment_user_id(instrument_id):
    resp = dp_client_mgmt.get_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=instrument_id,
        userId=USER_ID,
    )
    return resp["paymentInstrument"]["userId"]


try:
    # GetPaymentSession — spending-limit state and remaining spend.
    resp = dp_client_mgmt.get_payment_session(
        paymentManagerArn=MANAGER_ARN,
        paymentSessionId=PAYMENT_SESSION_ID,
        userId=USER_ID,
    )
    s = resp.get("paymentSession", {})
    limit_amt = s.get("limits", {}).get("maxSpendAmount", {})
    avail = s.get("availableLimits", {}).get("availableSpendAmount", {})
    print("── Payment session ──")
    print(f"  Spending limit: {limit_amt.get('value', '?')} {limit_amt.get('currency', '')}")
    print(f"  Remaining: {avail.get('value', '?')} {avail.get('currency', '')}")
    print(f"  Expires in: {s.get('expiryTimeInMinutes', '?')} minutes")

    # GetPaymentInstrumentBalance — on-chain USDC balance.
    payment_user_id = resolve_payment_user_id(PAYMENT_INSTRUMENT_ID)
    chain = "BASE" if PAYMENT_INSTRUMENT_NETWORK == "ETHEREUM" else "SOLANA"
    bal_resp = dp_client_mgmt.get_payment_instrument_balance(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
        userId=payment_user_id,
        chain=chain,
        token="USDC",
    )
    bal = bal_resp.get("tokenBalance", {})
    amount = int(bal.get("amount", "0")) / (10 ** int(bal.get("decimals", 6)))
    print(f"\n── Wallet balance ({chain}) ──")
    print(f"  Wallet:  {WALLET_ADDRESS}")
    print(f"  Balance: {amount:.6f} {bal.get('token', 'USDC')}")
except ClientError as exc:
    if exc.response.get("Error", {}).get("Code") == "ExpiredTokenException":
        raise SystemExit("STS session expired — re-run §5.1 to refresh clients, then re-run this cell.") from exc
    raise


# ── ListPaymentInstruments — every instrument under this connector ──
try:
    _li = dp_client_mgmt.list_payment_instruments(
        paymentManagerArn=MANAGER_ARN,
        paymentConnectorId=PAYMENT_CONNECTOR_ID,
        userId=resolve_payment_user_id(PAYMENT_INSTRUMENT_ID),
        maxResults=20,
    )
    _insts = _li.get("paymentInstruments", [])
    print(f"\n── Payment instruments ({len(_insts)}) ──")
    for _it in _insts:
        print(f"  {_it.get('paymentInstrumentId')}  ({_it.get('status', '?')})")
except ClientError as exc:
    print(f"\n  \u26a0\ufe0f  ListPaymentInstruments: {exc.response['Error']['Code']}")

# ── ListPaymentSessions — every session under this manager ──
try:
    _ls = dp_client_mgmt.list_payment_sessions(
        paymentManagerArn=MANAGER_ARN,
        userId=USER_ID,
        maxResults=20,
    )
    _sess = _ls.get("paymentSessions", [])
    print(f"\n── Payment sessions ({len(_sess)}) ──")
    for _se in _sess:
        print(f"  {_se.get('paymentSessionId')}  (expires in {_se.get('expiryTimeInMinutes', '?')} min)")
except ClientError as exc:
    print(f"\n  \u26a0\ufe0f  ListPaymentSessions: {exc.response['Error']['Code']}")


## 10. Cleanup

> ⚠️ **Cost notice:** the resources below incur AWS charges while running, and
> the live external x402 services settle real USDC. Run these cells in order to
> remove them and stop incurring charges.

### 10.1 Tear down AgentCore payments resources

Order matters: revoke the session, soft-delete the instrument, then the
connector, manager, and credential provider.


In [ ]:
# §10.1 is self-contained: it re-reads .env and rebuilds the control-plane and
# management clients if the kernel does not already have them, so cleanup works
# even after a restart. Deletes in dependency order: session -> instrument ->
# connector -> manager -> credential provider.
import os

import boto3
import botocore.exceptions
from boto3.session import Session
from dotenv import load_dotenv

from utils import assume_role, write_env_updates

load_dotenv(override=True)
AWS_REGION = os.environ.get("AWS_REGION", "us-west-2")
CONTROL_PLANE_ROLE_ARN = os.environ.get("CONTROL_PLANE_ROLE_ARN", "")
MANAGEMENT_ROLE_ARN = os.environ.get("MANAGEMENT_ROLE_ARN", "")
MANAGER_ARN = os.environ.get("MANAGER_ARN", "")
MANAGER_ID = MANAGER_ARN.rsplit("/", 1)[-1] if MANAGER_ARN else ""
PAYMENT_CONNECTOR_ID = os.environ.get("PAYMENT_CONNECTOR_ID", "")
CREDENTIAL_PROVIDER_ARN = os.environ.get("CREDENTIAL_PROVIDER_ARN", "")
PAYMENT_INSTRUMENT_ID = os.environ.get("PAYMENT_INSTRUMENT_ID", "")
PAYMENT_SESSION_ID = os.environ.get("PAYMENT_SESSION_ID", "")
USER_ID = os.environ.get("USER_ID", "")

import uuid as _uuid


def _client_token():
    return f"{_uuid.uuid4()}-{_uuid.uuid4().hex[:8]}"


# Rebuild clients if this cell is run in a fresh kernel.
try:
    cp_client
except NameError:
    cp_client = assume_role(
        Session(region_name=AWS_REGION), CONTROL_PLANE_ROLE_ARN, "x402-secure-cp-cleanup"
    ).client("bedrock-agentcore-control")
try:
    dp_client_mgmt
except NameError:
    dp_client_mgmt = assume_role(
        Session(region_name=AWS_REGION), MANAGEMENT_ROLE_ARN, "x402-secure-mgmt-cleanup"
    ).client("bedrock-agentcore")


def resolve_payment_user_id(instrument_id):
    resp = dp_client_mgmt.get_payment_instrument(
        paymentManagerArn=MANAGER_ARN,
        paymentInstrumentId=instrument_id,
        userId=USER_ID,
    )
    return resp["paymentInstrument"]["userId"]


def _safe_delete(fn, label, **kwargs):
    try:
        fn(**kwargs)
        print(f"  ✅ Deleted: {label}")
    except botocore.exceptions.ClientError as exc:
        code = exc.response["Error"]["Code"]
        msg = exc.response["Error"].get("Message", "")
        if code == "ResourceNotFoundException" or (code == "AccessDeniedException" and "not found" in msg.lower()):
            print(f"  ⚠️  Not found: {label}")
        else:
            raise


if not MANAGER_ARN:
    print("ℹ️  Nothing to tear down — MANAGER_ARN is unset in .env.")
else:
    payment_user_id = USER_ID
    try:
        payment_user_id = resolve_payment_user_id(PAYMENT_INSTRUMENT_ID)
    except Exception:  # noqa: BLE001 — best effort; fall back to operator USER_ID
        pass

    if PAYMENT_SESSION_ID:
        _safe_delete(
            dp_client_mgmt.delete_payment_session,
            f"Session {PAYMENT_SESSION_ID}",
            paymentManagerArn=MANAGER_ARN,
            paymentSessionId=PAYMENT_SESSION_ID,
            userId=payment_user_id,
        )
    if PAYMENT_INSTRUMENT_ID and PAYMENT_CONNECTOR_ID:
        _safe_delete(
            dp_client_mgmt.delete_payment_instrument,
            f"Instrument {PAYMENT_INSTRUMENT_ID}",
            paymentManagerArn=MANAGER_ARN,
            paymentConnectorId=PAYMENT_CONNECTOR_ID,
            paymentInstrumentId=PAYMENT_INSTRUMENT_ID,
            userId=payment_user_id,
        )
    if PAYMENT_CONNECTOR_ID:
        _safe_delete(
            cp_client.delete_payment_connector,
            f"Connector {PAYMENT_CONNECTOR_ID}",
            paymentManagerId=MANAGER_ID,
            paymentConnectorId=PAYMENT_CONNECTOR_ID,
            clientToken=_client_token(),
        )
    _safe_delete(
        cp_client.delete_payment_manager,
        f"Manager {MANAGER_ID}",
        paymentManagerId=MANAGER_ID,
        clientToken=_client_token(),
    )
    if CREDENTIAL_PROVIDER_ARN:
        cred_name = CREDENTIAL_PROVIDER_ARN.rsplit("/", 1)[-1]
        _safe_delete(
            cp_client.delete_payment_credential_provider,
            f"Credential provider {cred_name}",
            name=cred_name,
        )

    write_env_updates({
        "MANAGER_ARN": "",
        "PAYMENT_CONNECTOR_ID": "",
        "CREDENTIAL_PROVIDER_ARN": "",
        "PAYMENT_INSTRUMENT_ID": "",
        "PAYMENT_SESSION_ID": "",
        "WALLET_ADDRESS": "",
    })
    print("\n✅ AgentCore payments resources cleaned up.")
    print("💾 .env cleared of Manager/Connector/Instrument/Session IDs.")


### 10.2 Tear down the agent runtime

Only if you deployed the agent to AgentCore Runtime in §8. Skip this if you ran
the agent locally only.


In [ ]:
!bash test/integration/destroy-agent.sh

### 10.3 Remove local build artifacts

In [ ]:
import pathlib
import shutil

ROOT = pathlib.Path(".").resolve()
targets = [
    ROOT / "agent" / "cdk" / ".venv",
    ROOT / "agent" / "cdk" / "cdk.out",
    ROOT / "agent" / "cdk" / "outputs.json",
    ROOT / "agent" / "cdk" / "__pycache__",
    ROOT / "agent" / "container" / "__pycache__",
    ROOT / "__pycache__",
]
for path in targets:
    if not path.exists():
        print(f"  ↷ skip (absent): {path.relative_to(ROOT)}")
        continue
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()
    print(f"  🗑️  removed: {path.relative_to(ROOT)}")
print("\n✅ Local CDK artifacts removed.")

## Next Steps

Go deeper with the public AgentCore payments documentation:

- [AgentCore payments overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments.html)
- [How it works](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-how-it-works.html)
- [Core concepts](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-concepts.html)
- [Process a payment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-process-payment.html) — plugin reference, interrupt contract, network preferences
- [Agents that transact (announcement blog)](https://aws.amazon.com/blogs/machine-learning/agents-that-transact-introducing-amazon-bedrock-agentcore-payments-built-with-coinbase-and-stripe/)


# Congratulations!

You have built an agent that pays for a paid x402 service **safely** — gating
every target payment behind an independent, paid t54 x402-secure trust check
using **Amazon Bedrock AgentCore payments** — first as a local Strands agent,
then packaged into a container and deployed to **AgentCore Runtime**.

Here is what we covered:

* **Trust before spend** — the agent pays a scoring service to vet an endpoint before paying it, and the decision is enforced in code by `TrustedX402ServiceGateway`, not by the prompt.
* **Self-contained setup** — the notebook provisioned a full AgentCore payments stack inline (Credential Provider → Manager → Connector → Instrument → Session).
* **IAM role separation** — `ManagementRole` creates sessions; `ProcessPaymentRole` signs payments (enforced by IAM `Deny`, not documentation).
* **Per-invocation payment context** — the runtime holds no payment identifiers; the caller supplies them on each request, so one deployment serves many users.
* **Spending limit enforcement + audit** — `maxSpendAmount` caps both paid calls, and `GetPaymentSession` shows exactly what the agent spent.

**Ideas to extend this use case:**

* Register additional target x402 services in `agent/container/x402_service_registry.py` — the trust gate applies to each uniformly.
* Tighten `X402_TRUST_THRESHOLD` and add your own post-trust policy checks.
* Add AgentCore Memory so the agent remembers prior trust decisions across a session.
